In [1]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam


In [2]:
df = pd.read_csv(r'C:\Users\YASHVIR\OneDrive\Attachments\PROJECTS\NLP\imdb\dataset\cleaned_data.csv')
print(f"Loaded {len(df)} cleaned rows")
 
X = df['clean_review'].astype(str)
y = df['label']

Loaded 49582 cleaned rows


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Bidirectional, Dense, Dropout, SpatialDropout1D)

In [4]:
X_train, X_test,y_train,y_test = train_test_split(X,y, test_size=0.25, random_state=42, stratify=y)


In [5]:
lengths = X_train.str.split().apply(len)
print(lengths.describe())
print("95th percentile:", lengths.quantile(0.95))

count    37186.000000
mean       119.310574
std         89.924950
min          3.000000
25%         64.000000
50%         89.000000
75%        145.000000
max       1428.000000
Name: clean_review, dtype: float64
95th percentile: 307.0


In [6]:
MAX_WORDS = 15000
MAX_LEN = 307
 
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
 
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
 
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")
 

In [7]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
class_weight_dict = dict(enumerate(class_weights))


In [8]:
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=100),
    SpatialDropout1D(0.4),
    Bidirectional(LSTM(32, return_sequences=False)),
    Dropout(0.5),
    Dense(1, activation="sigmoid"),
])
 
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
 
model.summary()
 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
    ModelCheckpoint("best_lstm_model.keras", monitor="val_loss", save_best_only=True),
]
 

In [10]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=15,                 # EarlyStopping will cut this short if it plateaus
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)
 

Epoch 1/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 63s 129ms/step - accuracy: 0.8082 - loss: 0.4158 - val_accuracy: 0.8783 - val_loss: 0.3026 - learning_rate: 0.0010
Epoch 2/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 58s 125ms/step - accuracy: 0.9092 - loss: 0.2468 - val_accuracy: 0.8833 - val_loss: 0.3306 - learning_rate: 0.0010
Epoch 3/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 65s 139ms/step - accuracy: 0.9413 - loss: 0.1692 - val_accuracy: 0.8830 - val_loss: 0.3720 - learning_rate: 5.0000e-04
Epoch 4/15
465/465 ━━━━━━━━━━━━━━━━━━━━ 73s 157ms/step - accuracy: 0.9592 - loss: 0.1249 - val_accuracy: 0.8816 - val_loss: 0.3685 - learning_rate: 2.5000e-04


In [11]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()
 
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))
 

388/388 ━━━━━━━━━━━━━━━━━━━━ 14s 36ms/step

Accuracy: 0.8779444982252339
              precision    recall  f1-score   support

    Negative       0.88      0.87      0.88      6175
    Positive       0.87      0.88      0.88      6221

    accuracy                           0.88     12396
   macro avg       0.88      0.88      0.88     12396
weighted avg       0.88      0.88      0.88     12396



In [12]:
model.save('model.keras')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
 
print("\nSaved model.keras and tokenizer.pkl")


Saved model.keras and tokenizer.pkl
